# A trained 100-dimensional PINN as a certification benchmark

This notebook replaces the random target network by a standard physics-informed neural network for the manufactured Poisson problem

\[
-\Delta u=f\quad\text{in }\Omega=[-0.1,0.1]^{100},
\qquad u=g\quad\text{on }\partial\Omega.
\]

The exact solution is a nonconstant two-direction ridge function,

\[
u_*(x)=\sin(2.5\,a^\top x)+0.35\cos(1.75\,b^\top x),
\]

where $a,b\in\mathbb R^{100}$ are orthonormal dense directions. Hence

\[
f(x)=2.5^2\sin(2.5\,a^\top x)
+0.35\,1.75^2\cos(1.75\,b^\top x),
\qquad g=u_*|_{\partial\Omega}.
\]

The target architecture is exactly $100$-$50$-$50$-$50$-$1$ with tanh activations. The checkpoint was trained from the interior PDE residual and sampled Dirichlet boundary loss only; the exact solution is used for validation, not as supervised training data.

The half-width $0.1$ is an experimental scaling choice, not part of the Poisson equation itself. Because $a$ and $b$ are unit vectors, $a^\top x$ and $b^\top x$ can range on the order of one on this cube, so the two ridge modes remain nontrivial while the tanh preactivations stay in a range where a single-cell affine enclosure is still usable. On $[-1,1]^{100}$ the same frequencies would traverse much wider phase and preactivation intervals, making both ordinary PINN training and a global single-cell certificate substantially harder. The price of the small cube is $|\Omega|=0.2^{100}$, which is why raw norms are tiny and volume-normalized norms are also reported.

Certification compares interval arithmetic with the certified polynomial-Jacobian reductions Top-$k$, degree-capped Top-$k$, and coefficient-space PCA. The unreduced polynomial endpoint is structurally infeasible on the target architecture and is therefore not launched accidentally; reduced terms are always absorbed into a rigorously propagated pointwise remainder.

In [1]:
from __future__ import annotations

import math
import random
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch import nn

repo_root = Path.cwd()
while not (repo_root / 'src' / 'intervalnets').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from intervalnets import (
    IntervalTensor,
    PZIntegrationCell,
    enable_interval_eval,
    integrate_pz_onejet_squared,
    integrate_pz_value_squared,
    load_tanh_mlp_checkpoint,
    sequential_value_jacobian_laplacian,
)

torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)
enable_interval_eval()

DIM = 100
HALF_WIDTH = 0.1
HIDDEN = (50, 50, 50)
SEED = 20260731
K1 = 2.5
K2 = 1.75
COS_AMPLITUDE = 0.35
CHECKPOINT = repo_root / 'notebooks' / 'checkpoints' / 'pinn_100d_poisson.pt'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'torch={torch.__version__}, dtype={torch.get_default_dtype()}, threads={torch.get_num_threads()}')
print(f'checkpoint={CHECKPOINT.relative_to(repo_root)}')

torch=2.13.0+cu130, dtype=torch.float64, threads=1
checkpoint=notebooks/checkpoints/pinn_100d_poisson.pt


## PDE, model, and efficient PINN residual

In [2]:
def dense_directions(dim=DIM):
    a = torch.ones(dim)
    a /= torch.linalg.vector_norm(a)
    b = torch.tensor([1.0 if i % 2 == 0 else -1.0 for i in range(dim)])
    b -= torch.dot(a, b) * a
    b /= torch.linalg.vector_norm(b)
    return a, b


A, B = dense_directions()


def exact_solution(x):
    return (torch.sin(K1 * (x @ A)) + COS_AMPLITUDE * torch.cos(K2 * (x @ B))).unsqueeze(-1)


def exact_gradient(x):
    s = x @ A
    t = x @ B
    return K1 * torch.cos(K1 * s).unsqueeze(-1) * A - COS_AMPLITUDE * K2 * torch.sin(K2 * t).unsqueeze(-1) * B


def forcing(x):
    return (K1**2 * torch.sin(K1 * (x @ A)) + COS_AMPLITUDE * K2**2 * torch.cos(K2 * (x @ B))).unsqueeze(-1)


def make_model():
    layers, previous = [], DIM
    for width in HIDDEN:
        layers.extend([nn.Linear(previous, width), nn.Tanh()])
        previous = width
    layers.append(nn.Linear(previous, 1))
    model = nn.Sequential(*layers)
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    return model


def sample_interior(n, generator):
    return (2.0 * torch.rand((n, DIM), generator=generator) - 1.0) * HALF_WIDTH


def sample_boundary(n, generator):
    x = sample_interior(n, generator)
    coordinate = torch.randint(DIM, (n,), generator=generator)
    sign = torch.where(torch.rand(n, generator=generator) < 0.5, -1.0, 1.0)
    x[torch.arange(n), coordinate] = HALF_WIDTH * sign
    return x


def pinn_residual(model, x):
    value, jacobian, laplacian = sequential_value_jacobian_laplacian(model, x)
    return value, jacobian, -laplacian - forcing(x)


model = make_model()
sum(parameter.numel() for parameter in model.parameters()), model

(10201, Sequential(
  (0): Linear(in_features=100, out_features=50, bias=True)
  (1): Tanh()
  (2): Linear(in_features=50, out_features=50, bias=True)
  (3): Tanh()
  (4): Linear(in_features=50, out_features=50, bias=True)
  (5): Tanh()
  (6): Linear(in_features=50, out_features=1, bias=True)
))

## Reproducible PINN training

Set `RETRAIN = True` to regenerate the checkpoint. The default loads the included deterministic checkpoint, so certification can be rerun in seconds. Training uses Adam with 512 fresh interior and 512 fresh boundary points per step and the ordinary loss

\[
\mathcal L(\theta)=\mathbb E_\Omega|{-\Delta u_\theta-f}|^2
+20\,\mathbb E_{\partial\Omega}|u_\theta-g|^2.
\]

The Laplacian helper propagates the exact Hessian trace through the tanh MLP and remains differentiable with respect to its parameters; it changes computational organization, not the PINN objective.

In [3]:
def train_pinn(model, steps=1500, batch_size=512, lr=2e-3):
    generator = torch.Generator().manual_seed(SEED + 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=2e-4)
    history = []
    model.train()
    for step in range(1, steps + 1):
        interior = sample_interior(batch_size, generator)
        boundary = sample_boundary(batch_size, generator)
        _, _, residual = pinn_residual(model, interior)
        boundary_error = model(boundary) - exact_solution(boundary)
        residual_loss = residual.square().mean()
        boundary_loss = boundary_error.square().mean()
        loss = residual_loss + 20.0 * boundary_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        scheduler.step()
        if step == 1 or step % 100 == 0:
            history.append({
                'step': step,
                'loss': float(loss.detach()),
                'residual_loss': float(residual_loss.detach()),
                'boundary_loss': float(boundary_loss.detach()),
            })
    model.eval()
    return history


RETRAIN = False
if RETRAIN:
    training_history = train_pinn(model)
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'state_dict': model.state_dict(), 'seed': SEED}, CHECKPOINT)
else:
    model = load_tanh_mlp_checkpoint(CHECKPOINT)
    training_history = []

{'loaded_checkpoint': not RETRAIN, 'training_records': training_history[-3:]}

{'loaded_checkpoint': True, 'training_records': []}

## Candidate-network validation

In [4]:
validation_generator = torch.Generator().manual_seed(SEED + 222)
interior = sample_interior(8192, validation_generator)
boundary = sample_boundary(8192, validation_generator)
with torch.no_grad():
    prediction, network_jacobian, residual = pinn_residual(model, interior)
    target = exact_solution(interior)
    target_gradient = exact_gradient(interior)
    boundary_error = model(boundary) - exact_solution(boundary)
    error = prediction - target
    empirical_network_l2 = prediction.square().mean().sqrt()
    empirical_network_w12 = (prediction.square() + network_jacobian.square().sum(dim=(-2, -1), keepdim=True)).mean().sqrt()
    empirical_exact_l2 = target.square().mean().sqrt()
    empirical_exact_w12 = (target.square() + target_gradient.square().sum(dim=-1, keepdim=True)).mean().sqrt()

validation = {
    'solution_RMSE': float(error.square().mean().sqrt()),
    'solution_relative_L2_error': float(error.square().mean().sqrt() / target.square().mean().sqrt()),
    'solution_max_sample_error': float(error.abs().max()),
    'PDE_residual_RMSE': float(residual.square().mean().sqrt()),
    'boundary_RMSE': float(boundary_error.square().mean().sqrt()),
    'network_normalized_L2_MC': float(empirical_network_l2),
    'network_normalized_W12_MC': float(empirical_network_w12),
    'exact_normalized_L2_MC': float(empirical_exact_l2),
    'exact_normalized_W12_MC': float(empirical_exact_w12),
}
validation

{'solution_RMSE': 0.002859142422545456, 'solution_relative_L2_error': 0.007575039840436495, 'solution_max_sample_error': 0.030369810860190305, 'PDE_residual_RMSE': 0.01780581896203556, 'boundary_RMSE': 0.002867007950069476, 'network_normalized_L2_MC': 0.37731984665672036, 'network_normalized_W12_MC': 2.504706918197497, 'exact_normalized_L2_MC': 0.3774425590850363, 'exact_normalized_W12_MC': 2.5036437778283043}

## Certification diagnostics

The raw norm scales like $|\Omega|^{1/2}=0.2^{50}$, so both raw and volume-normalized intervals are reported. Volume normalization is not relative error: it only removes the factor $|\Omega|^{1/2}$. For an interval $[L,U]$, absolute width is $U-L$ and relative width is $(U-L)/\max(|L|,|U|)$ when the denominator is nonzero; the lower bound is not used as the denominator.

Before integration, the full certified Jacobian enclosure is summarized by mean component width, maximum component width, and the mean entrywise relative width

\[
\frac1N\sum_{ij}\frac{\overline J_{ij}-\underline J_{ij}}
{\max(|\underline J_{ij}|,|\overline J_{ij}|)},
\]

with exact-zero entries assigned zero. This relative width lies in $[0,2]$.

In [5]:
SQRT_VOLUME = (2.0 * HALF_WIDTH) ** (DIM / 2.0)
DOMAIN = IntervalTensor.from_bounds([-HALF_WIDTH] * DIM, [HALF_WIDTH] * DIM)


def norm_interval(squared):
    return math.sqrt(max(0.0, float(squared.lower))), math.sqrt(max(0.0, float(squared.upper)))


def interval_metrics(bounds, prefix):
    lower, upper = map(float, bounds)
    width = upper - lower
    return {
        f'{prefix}_lower': lower,
        f'{prefix}_upper': upper,
        f'{prefix}_absolute_width': width,
        f'{prefix}_relative_width': width / max(abs(lower), abs(upper)) if max(abs(lower), abs(upper)) > 0.0 else 0.0,
        f'{prefix}_normalized_lower': lower / SQRT_VOLUME,
        f'{prefix}_normalized_upper': upper / SQRT_VOLUME,
        f'{prefix}_normalized_absolute_width': width / SQRT_VOLUME,
    }


def jacobian_width_metrics(enclosure):
    lower = torch.as_tensor(enclosure.lower)
    upper = torch.as_tensor(enclosure.upper)
    widths = upper - lower
    scales = torch.maximum(lower.abs(), upper.abs())
    relative_widths = torch.where(scales > 0.0, widths / scales, 0.0)
    return {
        'J_mean_component_width_before_integration': float(widths.mean()),
        'J_max_component_width_before_integration': float(widths.max()),
        'J_relative_mean_component_width_before_integration': float(relative_widths.mean()),
    }


def benchmark_polynomial(model, strategy='topk', **kwargs):
    cell = PZIntegrationCell.from_affine_box(DOMAIN)
    start = perf_counter()
    traced = model.eval_pz_onejet(
        cell.domain, return_trace=True, reduction_strategy=strategy, **kwargs
    )
    forward_s = perf_counter() - start
    jacobian_metrics = jacobian_width_metrics(traced.final.J.interval_enclosure())
    start = perf_counter()
    l2_integrated_pz = integrate_pz_value_squared(traced.final.Y, cell, output='pz')
    l2_squared = l2_integrated_pz.interval_enclosure()
    l2_integration_s = perf_counter() - start
    start = perf_counter()
    w12_integrated_pz = integrate_pz_onejet_squared(traced.final, cell, output='pz')
    w12_squared = w12_integrated_pz.interval_enclosure()
    w12_integration_s = perf_counter() - start
    return {
        'strategy': strategy,
        **kwargs,
        'forward_s': forward_s,
        'L2_integration_s': l2_integration_s,
        'W12_integration_s': w12_integration_s,
        'total_W12_s': forward_s + w12_integration_s,
        'J_terms': len(traced.final.J.terms),
        'J_degree': max(map(sum, traced.final.J.terms), default=0),
        'noise_count': traced.final.J.num_noise,
        'L2_integrated_PZ_terms': len(l2_integrated_pz.terms),
        'W12_integrated_PZ_terms': len(w12_integrated_pz.terms),
        'W12_integrated_PZ_noise': w12_integrated_pz.num_noise,
        **jacobian_metrics,
        **interval_metrics(norm_interval(l2_squared), 'L2'),
        **interval_metrics(norm_interval(w12_squared), 'W12'),
        'trace': traced.records,
    }


SQRT_VOLUME

1.1258999068426271e-35

## Interval and certified polynomial-reduction benchmarks

In [6]:
configurations = [
    ('topk-32', 'topk', dict(max_terms=32)),
    ('topk-64', 'topk', dict(max_terms=64)),
    ('topk-96', 'topk', dict(max_terms=96)),
    ('topk-128', 'topk', dict(max_terms=128)),
    ('topk-192', 'topk', dict(max_terms=192)),
    ('degree-64', 'degree', dict(max_terms=64, max_degree=2)),
    ('pca-64', 'pca', dict(max_terms=64, pca_rank=4, pca_candidates=32)),
]

polynomial_rows = []
for label, strategy, kwargs in configurations:
    row = benchmark_polynomial(model, strategy=strategy, **kwargs)
    row['method'] = label
    polynomial_rows.append(row)

start = perf_counter()
interval_w12 = model.sobolev_norm(DOMAIN, p=2.0, order=1, method='interval')
interval_total_s = perf_counter() - start
interval_l2 = model.lpnorm(DOMAIN, p=2.0, method='interval')
interval_jacobian = model.eval_jacobian(DOMAIN)
interval_row = {
    'method': 'interval',
    'total_W12_s': interval_total_s,
    **jacobian_width_metrics(interval_jacobian),
    **interval_metrics((interval_l2.lower, interval_l2.upper), 'L2'),
    **interval_metrics((interval_w12.lower, interval_w12.upper), 'W12'),
}

assert all(row['total_W12_s'] < 3.0 for row in polynomial_rows)
benchmark_rows = [interval_row, *polynomial_rows]

columns = [
    'method', 'total_W12_s',
    'L2_normalized_lower', 'L2_normalized_upper', 'L2_normalized_absolute_width', 'L2_relative_width',
    'W12_normalized_lower', 'W12_normalized_upper', 'W12_normalized_absolute_width', 'W12_relative_width',
    'J_mean_component_width_before_integration', 'J_max_component_width_before_integration',
    'J_relative_mean_component_width_before_integration', 'J_terms', 'J_degree',
]
[{key: row.get(key) for key in columns} for row in benchmark_rows]

[{'method': 'interval', 'total_W12_s': 0.18789246800042747, 'L2_normalized_lower': 0.0, 'L2_normalized_upper': 7.979522775780479, 'L2_normalized_absolute_width': 7.979522775780479, 'L2_relative_width': 1.0, 'W12_normalized_lower': 0.0, 'W12_normalized_upper': 88.8468047131762, 'W12_normalized_absolute_width': 88.8468047131762, 'W12_relative_width': 1.0, 'J_mean_component_width_before_integration': 17.548814954264703, 'J_max_component_width_before_integration': 20.8648129804914, 'J_relative_mean_component_width_before_integration': 1.9897599352068311, 'J_terms': None, 'J_degree': None}, {'method': 'topk-32', 'total_W12_s': 1.3675986349990126, 'L2_normalized_lower': 0.0, 'L2_normalized_upper': 2.6768306182215453, 'L2_normalized_absolute_width': 2.6768306182215453, 'L2_relative_width': 1.0, 'W12_normalized_lower': 0.0, 'W12_normalized_upper': 86.68683003796212, 'W12_normalized_absolute_width': 86.68683003796212, 'W12_relative_width': 1.0, 'J_mean_component_width_before_integration': 17.11

### Raw norm intervals and absolute widths

In [7]:
raw_columns = [
    'method',
    'L2_lower', 'L2_upper', 'L2_absolute_width', 'L2_relative_width',
    'W12_lower', 'W12_upper', 'W12_absolute_width', 'W12_relative_width',
]
[{key: row.get(key) for key in raw_columns} for row in benchmark_rows]

[{'method': 'interval', 'L2_lower': 0.0, 'L2_upper': 8.984143949899863e-35, 'L2_absolute_width': 8.984143949899863e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 1.0003260914983017e-33, 'W12_absolute_width': 1.0003260914983017e-33, 'W12_relative_width': 1.0}, {'method': 'topk-32', 'L2_lower': 0.0, 'L2_upper': 3.0138433436891297e-35, 'L2_absolute_width': 3.0138433436891297e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 9.76006938642242e-34, 'W12_absolute_width': 9.76006938642242e-34, 'W12_relative_width': 1.0}, {'method': 'topk-64', 'L2_lower': 0.0, 'L2_upper': 3.0138433436891297e-35, 'L2_absolute_width': 3.0138433436891297e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 9.683503828321607e-34, 'W12_absolute_width': 9.683503828321607e-34, 'W12_relative_width': 1.0}, {'method': 'topk-96', 'L2_lower': 0.0, 'L2_upper': 3.0138433436891297e-35, 'L2_absolute_width': 3.0138433436891297e-35, 'L2_relative_width': 1.0, 'W12_lower': 0.0, 'W12_upper': 9.61

### Public default PZ APIs

In [8]:
start = perf_counter()
public_l2 = model.pz_l2norm(DOMAIN)
public_l2_s = perf_counter() - start
start = perf_counter()
public_w12 = model.pz_sobolev_norm(DOMAIN, order=1)
public_w12_s = perf_counter() - start

default_row = next(row for row in polynomial_rows if row['method'] == 'topk-96')
assert math.isclose(float(public_l2.upper), default_row['L2_upper'], rel_tol=1e-12)
assert math.isclose(float(public_w12.upper), default_row['W12_upper'], rel_tol=1e-12)
{
    'public_L2_s': public_l2_s,
    'public_W12_s': public_w12_s,
    **interval_metrics((public_l2.lower, public_l2.upper), 'public_L2'),
    **interval_metrics((public_w12.lower, public_w12.upper), 'public_W12'),
}

{'public_L2_s': 0.06607493099909334, 'public_W12_s': 2.057815976000711, 'public_L2_lower': -5e-324, 'public_L2_upper': 3.01384334368913e-35, 'public_L2_absolute_width': 3.01384334368913e-35, 'public_L2_relative_width': 1.0, 'public_L2_normalized_lower': -4.388184445513989e-289, 'public_L2_normalized_upper': 2.6768306182215458, 'public_L2_normalized_absolute_width': 2.6768306182215458, 'public_W12_lower': -5e-324, 'public_W12_upper': 9.611824696771708e-34, 'public_W12_absolute_width': 9.611824696771708e-34, 'public_W12_relative_width': 1.0, 'public_W12_normalized_lower': -4.388184445513989e-289, 'public_W12_normalized_upper': 85.37015269613308, 'public_W12_normalized_absolute_width': 85.37015269613308}

## Why the target unreduced polynomial is not executed

`reduction_strategy='none'` is a valid exact polynomial one-jet endpoint, but it is not a viable target-network benchmark. After the first tanh layer there are already roughly 100 domain-dependent derivative terms. The next chain-rule product couples these with about 150 derivative/value generators, producing on the order of $1.5\times10^4$ candidates; the third activation can then produce millions of candidates before canonicalization. Launching this path would violate the benchmark's memory and runtime purpose.

The unreduced endpoint remains covered by unit tests and by the small-network reference in `pz_w12_polynomial_reduction_benchmarks.ipynb`. Here, every target-network polynomial method is sound because the omitted tail is explicitly accumulated into a propagated pointwise remainder; no candidate term is simply dropped.

## What Top-$k$ reduction does

At each derivative-chain-rule product, every candidate monomial has a tensor coefficient $C_\alpha$. Top-$k$ scores it by its largest absolute component, keeps the $k$ highest-scoring exponent/coefficient pairs as dependent polynomial terms, and adds every discarded coefficient componentwise to a certified pointwise remainder radius. Equal exponent vectors are canonicalized before the final enclosure. Thus Top-$k$ is sound: it trades dependency information for a box remainder, but never deletes uncertainty. Larger $k$ preserves more correlation and cancellation, at the cost of more polynomial products and integration work.

PCA uses some of the discarded coefficient tensors differently: it retains a few shared coefficient-space directions and boxes only the orthogonal residual. This can preserve cancellation through later linear maps, although the benchmark below shows that the activation-approximation remainder, rather than the retained-support budget, dominates this particular PINN.

## Layer diagnostics for the default Top-96 method

For each hidden neuron, `tanh_prime_approximation_radius` is the certified coefficient $\delta_{\ell i}$ in the initial local enclosure

\[\tanh'(z_i)\in p_{\ell i}z_i+q_{\ell i}+\delta_{\ell i}[-1,1].\]

It is measured before multiplication by the incoming Jacobian and before any Top-$k$/PCA reduction, so it cleanly separates activation approximation error from compression error.

In [9]:
chosen = next(row for row in polynomial_rows if row['method'] == 'topk-96')
layer_diagnostics = [{
    'layer': record.layer_type,
    'seconds': record.elapsed_s,
    'Y_terms': record.summary['Y']['term_count'],
    'J_terms': record.summary['J']['term_count'],
    'J_degree': record.summary['J']['max_degree'],
    'J_remainder_mean_radius': record.summary['J']['remainder_mean_radius'],
    'J_remainder_max_radius': record.summary['J']['remainder_max_radius'],
} for record in chosen['trace']]
layer_diagnostics

[{'layer': 'Input', 'seconds': 0.0, 'Y_terms': 100, 'J_terms': 0, 'J_degree': 0, 'J_remainder_mean_radius': 0.0, 'J_remainder_max_radius': 0.0}, {'layer': 'Linear', 'seconds': 0.002622895999593311, 'Y_terms': 100, 'J_terms': 0, 'J_degree': 0, 'J_remainder_mean_radius': 0.0, 'J_remainder_max_radius': 0.0}, {'layer': 'Tanh', 'seconds': 0.02616049700009171, 'Y_terms': 150, 'J_terms': 96, 'J_degree': 1, 'J_remainder_mean_radius': 0.026446936553910286, 'J_remainder_max_radius': 0.07513935764656894}, {'layer': 'Linear', 'seconds': 0.009305661998951109, 'Y_terms': 150, 'J_terms': 96, 'J_degree': 1, 'J_remainder_mean_radius': 0.15294793951109945, 'J_remainder_max_radius': 0.23283848020502845}, {'layer': 'Tanh', 'seconds': 0.5622837949995301, 'Y_terms': 200, 'J_terms': 86, 'J_degree': 1, 'J_remainder_mean_radius': 0.17384077899371955, 'J_remainder_max_radius': 0.2799701365957077}, {'layer': 'Linear', 'seconds': 0.01211463400068169, 'Y_terms': 200, 'J_terms': 86, 'J_degree': 1, 'J_remainder_mean

### Initial per-neuron $\tanh'$ approximation-noise coefficients

In [ ]:
activation_records = [record for record in chosen['trace'] if record.layer_type == 'Tanh']
activation_error_columns = [
    record.summary['tanh_prime_approximation_radii'].detach().cpu().tolist()
    for record in activation_records
]
activation_error_summary = [{
    'hidden_layer': layer_index + 1,
    'min_delta': record.summary['tanh_prime_approximation_radius_min'],
    'mean_delta': record.summary['tanh_prime_approximation_radius_mean'],
    'max_delta': record.summary['tanh_prime_approximation_radius_max'],
} for layer_index, record in enumerate(activation_records)]
activation_error_rows = [{
    'neuron': neuron,
    **{f'hidden_layer_{layer + 1}_delta': values[neuron] for layer, values in enumerate(activation_error_columns)},
} for neuron in range(len(activation_error_columns[0]))]
activation_error_summary, activation_error_rows

## Interpretation

- The checkpoint is a meaningful PDE candidate: sampled relative solution error is below one percent, while residual and boundary errors are independently reported.
- All reduced polynomial methods meet the three-second target on one CPU thread.
- The final raw norms are extremely small only because $|\Omega|^{1/2}=0.2^{50}$. Volume-normalized bounds should be compared with the Monte Carlo RMS norms, but they are not relative errors.
- Standard PINN training does **not** automatically yield a certification-friendly parameterization. The sampled normalized $W^{1,2}$ norm is modest, but all single-cell certified lower bounds are zero and the upper bounds are much larger. The per-neuron $\tanh'$ residual table shows sizeable initial derivative-approximation radii in every hidden layer, and the accumulated Jacobian remainder is then amplified by later linear maps. This explains why preserving more Top-$k$ terms or PCA directions yields only a small improvement.
- Squared $L^2$ and $W^{1,2}$ integration returns a scalar PZ. Pointwise residual uncertainty is converted to a fresh integrated global generator, and only this final PZ is intervalized before the square root.
- Top-192 is the tightest configuration that robustly remains below three seconds here: its normalized $W^{1,2}$ interval is $[0,85.0694]$, versus $[0,85.3702]$ for Top-96 and $[0,88.8468]$ for intervals. The gain from 96 to 192 terms is only about $0.35\%$, so support growth is already saturating.
- This is an application-level finding: improving certificate-aware training, activation enclosures, or domain decomposition is more important here than simply increasing the retained support.
- Degree-64 coincides with Top-64 because the retained terms are degree one. PCA-64 gives only a small improvement relative to its added runtime. These outcomes are reported rather than selected away.